In [1]:
import random, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import librosa
from tqdm.notebook import tqdm

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.multioutput import MultiOutputClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

warnings.filterwarnings('ignore')
print('Libraries loaded')

Libraries loaded


In [2]:
RUN = "LOCAL"  # "LOCAL" | "KAGGLE"

SR         = 32000
DURATION   = 5
TARGET_LEN = SR * DURATION
N_MFCC     = 20
N_FEAT     = N_MFCC * 6 + 2
SEED       = 42

if RUN == "LOCAL":
    BASE_DIR   = Path("C:/Users/abdul/Downloads")
    FEAT_CACHE = Path("C:/Users/abdul/Downloads/features_v3.npz")
    CHUNK_DIR  = Path("C:/Users/abdul/Downloads")
else:
    BASE_DIR   = Path("/kaggle/input/competitions/birdclef-2026")
    FEAT_CACHE = Path("/kaggle/working/features_v4.npz")  # v4 = 122 features
    CHUNK_DIR  = Path("/kaggle/working")

TRAIN_AUDIO = BASE_DIR / "train_audio"
TRAIN_SC    = BASE_DIR / "train_soundscapes"
META_CSV    = BASE_DIR / "train.csv"
TAX_CSV     = BASE_DIR / "taxonomy.csv"
SC_CSV      = BASE_DIR / "train_soundscapes_labels.csv"

CHUNK_SIZE = 5000
SC_REPEATS = 5

random.seed(SEED)
np.random.seed(SEED)


In [3]:
df  = pd.read_csv(META_CSV)
tax = pd.read_csv(TAX_CSV)
sc  = pd.read_csv(SC_CSV)

species_list   = tax['primary_label'].astype(str).tolist()
species_to_idx = {sp: i for i, sp in enumerate(species_list)}
NUM_CLASSES    = len(species_list)

labels   = df['primary_label'].astype(str)
counts   = labels.value_counts()
valid_df = df[labels.isin(counts[counts >= 2].index)].reset_index(drop=True)

# split by recording to prevent leakage - same recording must not appear in both splits
val_files   = pd.Series(sc['filename'].unique()).sample(frac=0.2, random_state=SEED)
sc_val_df   = sc[sc['filename'].isin(val_files)]
sc_train_df = sc[~sc['filename'].isin(val_files)]

print(f'Species to predict  : {NUM_CLASSES}')
print(f'Training clips      : {len(valid_df)}')
print(f'Soundscape train    : {len(sc_train_df)}')
print(f'Soundscape val      : {len(sc_val_df)}')

Species to predict  : 234
Training clips      : 35545
Soundscape train    : 1184
Soundscape val      : 294


## Exploratory Data Analysis


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

class_counts = {'Aves': 34799, 'Amphibia': 451, 'Insecta': 199, 'Mammalia': 99, 'Reptilia': 1}
palette = ['#4e9a5e', '#e07b39', '#4a7db5', '#c95f5f', '#8e6bbf']
bars = axes[0].bar(list(class_counts.keys()), list(class_counts.values()),
                   color=palette, edgecolor='grey', linewidth=0.5)
axes[0].set_title('Training Clips by Animal Class')
axes[0].set_xlabel('Class')
axes[0].set_ylabel('Number of clips')
axes[0].grid(axis='y', alpha=0.3)
for bar, val in zip(bars, class_counts.values()):
    axes[0].text(bar.get_x() + bar.get_width() / 2, val + 150, str(val),
                 ha='center', va='bottom', fontsize=9)

top20 = df['primary_label'].value_counts().head(20)
axes[1].barh(range(20), top20.values[::-1], color='#9ecae1', edgecolor='grey', linewidth=0.5)
axes[1].set_yticks(range(20))
axes[1].set_yticklabels(top20.index[::-1], fontsize=8)
axes[1].set_title('Top 20 Species by Training Clip Count')
axes[1].set_xlabel('Number of clips')
axes[1].grid(axis='x', alpha=0.3)

plt.suptitle('Dataset Overview', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import librosa.display

_row   = valid_df.iloc[0]
_path  = str(TRAIN_AUDIO / _row['filename'])
_off   = max(0.0, (librosa.get_duration(path=_path) - DURATION) / 2.0)
_audio, _ = librosa.load(_path, sr=SR, offset=_off, duration=DURATION)

fig, axes = plt.subplots(3, 1, figsize=(12, 9))
fig.suptitle(f'Audio Sample -- {_row["primary_label"]}', fontsize=12)

axes[0].plot(np.linspace(0, DURATION, len(_audio)), _audio, linewidth=0.6, color='steelblue')
axes[0].set_title('Waveform')
axes[0].set_xlabel('Time (s)')
axes[0].set_ylabel('Amplitude')
axes[0].grid(alpha=0.3)

_mel    = librosa.feature.melspectrogram(y=_audio, sr=SR, n_mels=128, fmin=50, fmax=14000)
_mel_db = librosa.power_to_db(_mel, ref=np.max)
_img    = librosa.display.specshow(_mel_db, sr=SR, hop_length=512,
                                    x_axis='time', y_axis='mel',
                                    fmin=50, fmax=14000, ax=axes[1])
axes[1].set_title('Mel Spectrogram')
fig.colorbar(_img, ax=axes[1], format='%+2.0f dB')

_mfcc = librosa.feature.mfcc(y=_audio, sr=SR, n_mfcc=N_MFCC)
_img2 = librosa.display.specshow(_mfcc, sr=SR, hop_length=512, x_axis='time', ax=axes[2])
axes[2].set_title('MFCCs (20 coefficients)')
axes[2].set_ylabel('MFCC index')
fig.colorbar(_img2, ax=axes[2])

plt.tight_layout()
plt.show()


## Feature Extraction

20 MFCCs + 20 delta MFCCs x (mean + std + max) + spectral centroid + ZCR = **122 features**.
Adding max captures brief calls that get diluted by mean averaging in overlapping soundscapes.

In [4]:
def extract_features(audio):
    if len(audio) < TARGET_LEN:
        audio = np.pad(audio, (0, TARGET_LEN - len(audio)))
    audio = audio[:TARGET_LEN]

    mfcc  = librosa.feature.mfcc(y=audio, sr=SR, n_mfcc=N_MFCC)
    delta = librosa.feature.delta(mfcc)
    cent  = librosa.feature.spectral_centroid(y=audio, sr=SR)
    zcr   = librosa.feature.zero_crossing_rate(y=audio)

    return np.concatenate([
        mfcc.mean(axis=1),    # 20  spectral envelope mean
        mfcc.std(axis=1),     # 20  envelope variability
        mfcc.max(axis=1),     # 20  peak shape - catches brief calls in overlapping audio
        delta.mean(axis=1),   # 20  how spectrum changes over time
        delta.std(axis=1),    # 20  variability of temporal change
        delta.max(axis=1),    # 20  peak temporal change
        cent.mean(axis=1),    #  1  brightness
        zcr.mean(axis=1),     #  1  noisiness
    ]).astype(np.float32)


In [5]:
def _sc_row_to_target(row):
    target = np.zeros(NUM_CLASSES, dtype=np.float32)
    for sp in str(row['primary_label']).split(';'):
        sp = sp.strip()
        if sp in species_to_idx:
            target[species_to_idx[sp]] = 1.0
    return target

In [6]:
_from_cache = FEAT_CACHE.exists()

if _from_cache:
    print('Loading cached features...')
    data       = np.load(FEAT_CACHE)
    X_clips    = data['X_clips']
    y_clips    = data['y_clips']
    X_sc_all   = data['X_sc_all']
    Y_sc_all   = data['Y_sc_all']
    print(f'  Clips : {X_clips.shape}')
    print(f'  SC    : {X_sc_all.shape}')
else:
    # validate and resume from existing chunks
    chunk_files = sorted(CHUNK_DIR.glob('clips_chunk_*.npz'))
    valid_chunks = []
    for cf in chunk_files:
        try:
            d    = np.load(cf)
            X, y = d['X'], d['y']
            if X.shape[1] != N_FEAT or len(X) != len(y) or len(X) == 0 or np.isnan(X).any():
                raise ValueError('corrupt')
            valid_chunks.append((cf, len(y)))
        except Exception:
            for bad in [cf] + [f for f in chunk_files if f.name > cf.name]:
                bad.unlink(missing_ok=True)
            break

    resume_idx    = sum(count for _, count in valid_chunks)
    next_chunk_id = len(valid_chunks)
    if resume_idx > 0:
        print(f'Resuming from clip {resume_idx} ({next_chunk_id} valid chunks found)')
    else:
        print(f'Starting extraction from {len(valid_df)} clips...')

    t0 = time.time()
    X_batch, y_batch = [], []
    chunk_id = next_chunk_id
    for _, row in tqdm(valid_df.iloc[resume_idx:].iterrows(),
                       total=len(valid_df) - resume_idx, desc='Clips'):
        fp = TRAIN_AUDIO / row['filename']
        try:
            total    = librosa.get_duration(path=str(fp))
            offset   = max(0.0, (total - DURATION) / 2.0)
            audio, _ = librosa.load(str(fp), sr=SR, offset=offset, duration=DURATION)
            if np.abs(audio).max() < 1e-4:
                continue
            X_batch.append(extract_features(audio))
            y_batch.append(species_to_idx.get(str(row['primary_label']), -1))
        except Exception:
            continue
        if len(X_batch) == CHUNK_SIZE:
            np.savez(CHUNK_DIR / f'clips_chunk_{chunk_id:03d}.npz',
                     X=np.array(X_batch, dtype=np.float32),
                     y=np.array(y_batch, dtype=np.int32))
            X_batch, y_batch = [], []
            chunk_id += 1

    if X_batch:
        np.savez(CHUNK_DIR / f'clips_chunk_{chunk_id:03d}.npz',
                 X=np.array(X_batch, dtype=np.float32),
                 y=np.array(y_batch, dtype=np.int32))
        chunk_id += 1

    print(f'Extraction done: {chunk_id} chunks, {time.time() - t0:.0f}s')

Loading cached features...
  Clips : (35538, 82)
  SC    : (1478, 82)


In [7]:
if not _from_cache:
    chunk_files = sorted(CHUNK_DIR.glob('clips_chunk_*.npz'))
    X_parts, y_parts = [], []
    for f in chunk_files:
        d = np.load(f)
        X_parts.append(d['X'])
        y_parts.append(d['y'])

    X_clips = np.concatenate(X_parts, axis=0)
    y_clips = np.concatenate(y_parts, axis=0)

    mask    = y_clips >= 0
    X_clips = X_clips[mask]
    y_clips = y_clips[mask]

    for f in chunk_files:
        f.unlink()

    print(f'X_clips : {X_clips.shape}')
    print(f'y_clips : {y_clips.shape}')

In [8]:
if not _from_cache:
    t0 = time.time()
    print(f'Extracting features from {len(sc)} soundscape windows...')
    X_sc_rows, Y_sc_rows = [], []
    for _, row in tqdm(sc.iterrows(), total=len(sc), desc='Soundscapes'):
        parts     = str(row['start']).split(':')
        start_sec = int(parts[0]) * 3600 + int(parts[1]) * 60 + int(parts[2])
        try:
            audio, _ = librosa.load(str(TRAIN_SC / row['filename']), sr=SR, offset=float(start_sec), duration=DURATION)
            X_sc_rows.append(extract_features(audio))
            Y_sc_rows.append(_sc_row_to_target(row))
        except Exception:
            X_sc_rows.append(np.zeros(N_FEAT, dtype=np.float32))
            Y_sc_rows.append(np.zeros(NUM_CLASSES, dtype=np.float32))

    X_sc_all = np.array(X_sc_rows, dtype=np.float32)
    Y_sc_all = np.array(Y_sc_rows, dtype=np.float32)

    np.savez(FEAT_CACHE, X_clips=X_clips, y_clips=y_clips, X_sc_all=X_sc_all, Y_sc_all=Y_sc_all)
    print(f'Soundscapes done: {X_sc_all.shape} in {time.time() - t0:.0f}s')

In [9]:
val_pos   = sc_val_df.index.tolist()
train_pos = sc_train_df.index.tolist()

X_sc_val,   Y_sc_val   = X_sc_all[val_pos],   Y_sc_all[val_pos]
X_sc_train, Y_sc_train = X_sc_all[train_pos], Y_sc_all[train_pos]

Y_clips = np.eye(NUM_CLASSES, dtype=np.float32)[y_clips]

X_train = np.vstack([X_clips] + [X_sc_train] * SC_REPEATS)
Y_train = np.vstack([Y_clips]  + [Y_sc_train] * SC_REPEATS)

print(f'X_train : {X_train.shape}  (clips + soundscape x{SC_REPEATS})')
print(f'X_val   : {X_sc_val.shape}')

X_train : (41458, 82)  (clips + soundscape x5)
X_val   : (294, 82)


## Classifier Comparison

Three classifiers, Binary Relevance (one classifier per species), evaluated on held-out soundscape windows.

In [10]:
scaler    = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_val_s   = scaler.transform(X_sc_val)

sp_per_window = Y_sc_val.sum(axis=1)
print(f"Val windows          : {len(Y_sc_val)}")
print(f"Avg species / window : {sp_per_window.mean():.1f}  (max {int(sp_per_window.max())})")

active_cols = np.where(Y_train.max(axis=0) > 0)[0]
print(f"Species with training positives : {len(active_cols)} / {NUM_CLASSES}")


Val windows          : 294
Avg species / window : 3.9  (max 8)
Species with training positives : 230 / 234


In [11]:
class _FullPredictor:
    """Wraps a MultiOutputClassifier trained on active_cols, padding zeros for inactive species."""
    def __init__(self, clf, cols, n_total):
        self._clf = clf
        self._cols = cols
        self._n = n_total

    def predict_proba(self, X):
        sub = self._clf.predict_proba(X)
        out = [np.column_stack([np.ones(len(X)), np.zeros(len(X))]) for _ in range(self._n)]
        for i, p in zip(self._cols, sub):
            out[i] = p
        return out


def macro_auc(clf, X, Y):
    P_list = clf.predict_proba(X)
    P = np.zeros((len(X), NUM_CLASSES), dtype=np.float32)
    for i, p in enumerate(P_list):
        if p.shape[1] == 2:  # RF returns (N,1) for single-class columns
            P[:, i] = p[:, 1]
    aucs = [
        roc_auc_score(Y[:, i], P[:, i])
        for i in range(NUM_CLASSES)
        if 0 < Y[:, i].sum() < len(Y)
    ]
    return round(float(np.mean(aucs)), 4) if aucs else float('nan')


In [12]:
results = {}

In [13]:
print("Training Logistic Regression...")
t0 = time.time()

_lr_inner = MultiOutputClassifier(
    LogisticRegression(max_iter=1000, solver="lbfgs", class_weight="balanced", random_state=SEED), n_jobs=-1
)
_lr_inner.fit(X_train_s, Y_train[:, active_cols])
lr = _FullPredictor(_lr_inner, active_cols, NUM_CLASSES)
train_time = round(time.time() - t0, 1)
auc = macro_auc(lr, X_val_s, Y_sc_val)

results["Logistic Regression"] = {"Detection Score": auc, "Train time (s)": train_time}
print(f"  Train time       : {train_time}s")
print(f"  Detection Score  : {auc:.4f}")


Training Logistic Regression...


  Train time       : 43.4s
  Detection Score  : 0.7258


In [14]:
print("Training Random Forest  (native multi-output, 100 trees)...")
t0 = time.time()

rf = RandomForestClassifier(n_estimators=100, max_depth=15, min_samples_leaf=5, n_jobs=-1, random_state=SEED)
rf.fit(X_train, Y_train)
train_time = round(time.time() - t0, 1)
auc = macro_auc(rf, X_sc_val, Y_sc_val)

results["Random Forest"] = {"Detection Score": auc, "Train time (s)": train_time}

_has_max = X_train.shape[1] > 82
feat_names = (
    [f"MFCC{i}_mean"   for i in range(N_MFCC)] +
    [f"MFCC{i}_std"    for i in range(N_MFCC)] +
    ([f"MFCC{i}_max"   for i in range(N_MFCC)] if _has_max else []) +
    [f"dMFCC{i}_mean"  for i in range(N_MFCC)] +
    [f"dMFCC{i}_std"   for i in range(N_MFCC)] +
    ([f"dMFCC{i}_max"  for i in range(N_MFCC)] if _has_max else []) +
    ["centroid_mean", "zcr_mean"]
)
print(f"  Train time       : {train_time}s")
print(f"  Detection Score  : {auc:.4f}")
print("  Top-5 features:")
top5 = rf.feature_importances_.argsort()[::-1][:5]
for idx in top5:
    print(f"    {feat_names[idx]:20s}  {rf.feature_importances_[idx]:.4f}")


Training Random Forest  (native multi-output, 100 trees)...


  Train time       : 442.5s
  Detection Score  : 0.7564
  Top-5 features:
    MFCC0_mean            0.0855


    MFCC9_mean            0.0668
    MFCC15_mean           0.0465
    MFCC2_mean            0.0390


    MFCC5_mean            0.0375


In [15]:
print("Training SVM...")
t0 = time.time()

_svm_inner = MultiOutputClassifier(
    CalibratedClassifierCV(LinearSVC(max_iter=2000, random_state=SEED)), n_jobs=-1
)
_svm_inner.fit(X_train_s, Y_train[:, active_cols])
svm = _FullPredictor(_svm_inner, active_cols, NUM_CLASSES)
train_time = round(time.time() - t0, 1)
auc = macro_auc(svm, X_val_s, Y_sc_val)

results["Linear SVM"] = {"Detection Score": auc, "Train time (s)": train_time}
print(f"  Train time       : {train_time}s")
print(f"  Detection Score  : {auc:.4f}")


Training SVM...


  Train time       : 265.8s
  Detection Score  : 0.7087


In [16]:
df_results = pd.DataFrame(results).T.rename_axis("Classifier")
df_results = df_results.sort_values("Detection Score", ascending=False)

print(df_results.to_string())
print()
if df_results["Detection Score"].notna().any():
    best = df_results["Detection Score"].idxmax()
    print(f"Best: {best}  (Detection Score {df_results.loc[best, 'Detection Score']:.4f})")
else:
    print("No valid scores - check that the feature cache was built with full data.")


                     Detection Score  Train time (s)
Classifier                                          
Random Forest                 0.7564           442.5
Logistic Regression           0.7258            43.4
Linear SVM                    0.7087           265.8

Best: Random Forest  (Detection Score 0.7564)


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

_names  = list(df_results.index)
_scores = df_results['Detection Score'].tolist()
_colors = ['#4e9a5e' if s == max(_scores) else '#9ecae1' for s in _scores]
bars = axes[0].barh(_names, _scores, color=_colors, edgecolor='grey', linewidth=0.5)
axes[0].set_xlim(0.65, 0.80)
axes[0].set_xlabel('Detection Score (macro ROC-AUC)')
axes[0].set_title('Classifier Comparison')
axes[0].grid(axis='x', alpha=0.3)
for bar, val in zip(bars, _scores):
    axes[0].text(val + 0.001, bar.get_y() + bar.get_height() / 2,
                 f'{val:.4f}', va='center', fontsize=10)

_has_max = X_train.shape[1] > 82
_fnames  = (
    [f'MFCC{i}_mean'  for i in range(N_MFCC)] +
    [f'MFCC{i}_std'   for i in range(N_MFCC)] +
    ([f'MFCC{i}_max'  for i in range(N_MFCC)] if _has_max else []) +
    [f'dMFCC{i}_mean' for i in range(N_MFCC)] +
    [f'dMFCC{i}_std'  for i in range(N_MFCC)] +
    ([f'dMFCC{i}_max' for i in range(N_MFCC)] if _has_max else []) +
    ['centroid_mean', 'zcr_mean']
)
_top10_idx  = rf.feature_importances_.argsort()[::-1][:10]
_top10_name = [_fnames[i] for i in _top10_idx][::-1]
_top10_val  = [rf.feature_importances_[i] for i in _top10_idx][::-1]
axes[1].barh(_top10_name, _top10_val, color='#fdae6b', edgecolor='grey', linewidth=0.5)
axes[1].set_xlabel('Feature Importance (mean impurity decrease)')
axes[1].set_title('Random Forest -- Top-10 Feature Importances')
axes[1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()


## Cross-Validation -- Logistic Regression

A single 80/20 split only scores ~40 of 234 species - too narrow to trust.
5-fold recording-level CV evaluates each recording exactly once across 5 rounds,
covering more species and giving a stable estimate.
RF and SVM are excluded: too slow for 5 folds within the time budget.


In [17]:
from sklearn.model_selection import GroupKFold

gkf     = GroupKFold(n_splits=5)
cv_aucs = []

for fold, (tr_idx, val_idx) in enumerate(gkf.split(X_sc_all, groups=sc["filename"])):
    Xf_tr  = np.vstack([X_clips] + [X_sc_all[tr_idx]]  * SC_REPEATS)
    Yf_tr  = np.vstack([Y_clips]  + [Y_sc_all[tr_idx]]  * SC_REPEATS)
    Xf_val = X_sc_all[val_idx]
    Yf_val = Y_sc_all[val_idx]

    sf          = StandardScaler().fit(Xf_tr)
    active_cols = np.where(Yf_tr.max(axis=0) > 0)[0]
    n_sp        = int((Yf_val.sum(axis=0) > 0).sum())

    _lr_f = MultiOutputClassifier(
        LogisticRegression(max_iter=1000, solver="lbfgs", class_weight="balanced", random_state=SEED),
        n_jobs=-1
    )
    _lr_f.fit(sf.transform(Xf_tr), Yf_tr[:, active_cols])
    lr_f = _FullPredictor(_lr_f, active_cols, NUM_CLASSES)

    fold_score = macro_auc(lr_f, sf.transform(Xf_val), Yf_val)
    cv_aucs.append(fold_score)
    print(f"  Fold {fold+1}/5  Detection Score : {fold_score:.4f}  ({len(val_idx)} windows, {n_sp} species scored)")

print()
print(f"Logistic Regression  5-fold CV  Detection Score : {np.mean(cv_aucs):.4f}  +/-  {np.std(cv_aucs):.4f}")


  Fold 1/5  Detection Score : 0.7620  (298 windows, 32 species scored)


  Fold 2/5  Detection Score : 0.8495  (294 windows, 36 species scored)


  Fold 3/5  Detection Score : 0.7905  (296 windows, 36 species scored)


  Fold 4/5  Detection Score : 0.7251  (292 windows, 44 species scored)


  Fold 5/5  Detection Score : 0.8290  (298 windows, 35 species scored)

Logistic Regression  5-fold CV  Detection Score : 0.7912  +/-  0.0448


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4))
_mean       = np.mean(cv_aucs)
_bar_colors = ['#4e9a5e' if s >= _mean else '#fc8d59' for s in cv_aucs]
bars = ax.bar([f'Fold {i+1}' for i in range(len(cv_aucs))], cv_aucs,
              color=_bar_colors, edgecolor='grey', linewidth=0.5, width=0.5)
ax.axhline(_mean, color='steelblue', linewidth=1.8, linestyle='--',
           label=f'Mean = {_mean:.4f}  +/-  {np.std(cv_aucs):.4f}')
ax.set_ylim(0.65, 0.90)
ax.set_ylabel('Detection Score (macro ROC-AUC)')
ax.set_title('5-Fold Cross-Validation -- Logistic Regression')
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)
for bar, val in zip(bars, cv_aucs):
    ax.text(bar.get_x() + bar.get_width() / 2, val + 0.003,
            f'{val:.4f}', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.show()
